# Options Pricing & Implied Volatility Surface

This notebook walks through:
1. **Black-Scholes-Merton (BSM)** pricing and Greeks
2. **Heston stochastic volatility** model pricing
3. **Implied volatility surface** calibrated to live SPY options
4. **Delta-hedging simulation** with P&L attribution
5. **GARCH forecast integration** — risk premium vs vol surprise


In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Black-Scholes-Merton Model

BSM assumes the underlying follows geometric Brownian motion:
$$dS = \mu S\,dt + \sigma S\,dW$$

The European call price is:
$$C = S N(d_1) - K e^{-rT} N(d_2)$$

where $d_1 = \frac{\ln(S/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}$, $d_2 = d_1 - \sigma\sqrt{T}$.

**Limitation**: BSM uses a *single flat vol* — real markets show vol smiles and term structure.

In [ ]:
from pricing.bsm import bsm_price, greeks, implied_vol

S, K, T, r, sigma = 530.0, 530.0, 0.25, 0.05, 0.20
g = greeks(S, K, T, r, sigma, 'call')
print('BSM ATM Call Greeks:')
for name, val in g.items():
    print(f'  {name:8s}: {val:.6f}')

iv_check = implied_vol(g['price'], S, K, T, r, 'call')
print(f'\nImplied vol from price: {iv_check:.4f} (expected {sigma})')

In [ ]:
# Visualize option price and delta across strikes
strikes = np.linspace(430, 630, 100)
call_prices = [bsm_price(S, k, T, r, sigma, 'call') for k in strikes]
put_prices  = [bsm_price(S, k, T, r, sigma, 'put')  for k in strikes]

fig, (ax1, ax2) = plt.subplots(1, 2)
ax1.plot(strikes, call_prices, label='Call', color='steelblue')
ax1.plot(strikes, put_prices,  label='Put',  color='crimson')
ax1.axvline(S, color='gray', linestyle='--', label='ATM')
ax1.set_xlabel('Strike'); ax1.set_ylabel('Price ($)')
ax1.set_title('BSM Option Prices'); ax1.legend(); ax1.grid(alpha=0.3)

from pricing.bsm import delta as bsm_delta
call_deltas = [bsm_delta(S, k, T, r, sigma, 'call') for k in strikes]
put_deltas  = [bsm_delta(S, k, T, r, sigma, 'put')  for k in strikes]
ax2.plot(strikes, call_deltas, label='Call Δ', color='steelblue')
ax2.plot(strikes, put_deltas,  label='Put Δ',  color='crimson')
ax2.axvline(S, color='gray', linestyle='--')
ax2.set_xlabel('Strike'); ax2.set_ylabel('Delta')
ax2.set_title('BSM Delta'); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()

## 2. Heston Stochastic Volatility Model

Heston (1993) allows variance to be stochastic:
$$dS = rS\,dt + \sqrt{v}\,S\,dW_1$$
$$dv = \kappa(\theta - v)\,dt + \sigma_v\sqrt{v}\,dW_2, \quad \langle dW_1 dW_2 \rangle = \rho\,dt$$

Parameters:
- $v_0$: initial variance, $\kappa$: mean reversion speed, $\theta$: long-run variance
- $\sigma_v$: vol-of-vol, $\rho$: correlation (negative → negative skew)

Pricing uses the characteristic function and Fourier inversion.

**Key advantage over BSM**: produces realistic vol smiles and skews.

In [ ]:
from pricing.heston import heston_price

params = dict(v0=0.04, kappa=2.0, theta=0.04, sigma_v=0.3, rho=-0.7)
heston_atm = heston_price(S, K, T, r, **params)
bsm_atm    = bsm_price(S, K, T, r, sigma, 'call')
print(f'Heston ATM Call: {heston_atm:.4f}')
print(f'BSM    ATM Call: {bsm_atm:.4f}')

# Show how Heston produces a vol smile
heston_ivs = []
bsm_ivs    = []
strike_range = np.linspace(430, 630, 30)
for k in strike_range:
    hp = heston_price(S, k, T, r, **params)
    hiv = implied_vol(hp, S, k, T, r, 'call')
    heston_ivs.append(hiv)
    bsm_ivs.append(sigma)  # flat

plt.figure()
plt.plot(strike_range / S, np.array(heston_ivs)*100, label='Heston IV', color='crimson')
plt.plot(strike_range / S, np.array(bsm_ivs)*100,    label='BSM flat',  color='steelblue', linestyle='--')
plt.axvline(1.0, color='gray', linestyle=':')
plt.xlabel('Moneyness (K/S)'); plt.ylabel('Implied Vol (%)')
plt.title('Heston vs BSM Implied Vol Smile'); plt.legend(); plt.grid(alpha=0.3)

## 3. Heston Calibration to SPY Options

We minimize RMSE between Heston model IVs and market IVs across all strikes/expiries.

In [ ]:
from calibration.calibrate_heston import calibrate
params, rmse, iv_df, raw_data = calibrate()
print(f'\nCalibrated parameters: {params}')
print(f'RMSE: {rmse:.4f}')

In [ ]:
# Show market IV scatter
fig, ax = plt.subplots()
for exp in iv_df['expiry'].unique():
    sub = iv_df[iv_df['expiry'] == exp].sort_values('moneyness')
    days = int(sub['T'].mean() * 365)
    ax.scatter(sub['moneyness'], sub['iv']*100, s=15, label=f'{exp} ({days}d)', alpha=0.8)
ax.axvline(1.0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Moneyness (K/S)'); ax.set_ylabel('Implied Vol (%)')
ax.set_title('Market Implied Volatility by Expiry'); ax.legend(); ax.grid(alpha=0.3)

## 4. Implied Volatility Surface

In [ ]:
from IPython.display import Image
Image('../outputs/iv_surface.png')

In [ ]:
Image('../outputs/heston_vs_bsm_surface.png')

## 5. Delta-Hedging Simulation

We buy 1 ATM call and delta-hedge daily over 30 days.

**P&L Attribution**:
- **Gamma P&L**: $\frac{1}{2}\Gamma (\Delta S)^2$ — profit from convexity
- **Theta P&L**: $\Theta \cdot \Delta t$ — cost of holding the option
- In BSM world, $\Gamma$ and $\Theta$ cancel in expectation


In [ ]:
from hedging.delta_hedge import simulate_delta_hedge, print_summary
results = simulate_delta_hedge(n_paths=500)  # faster for notebook
print_summary(results)

In [ ]:
Image('../outputs/delta_hedge_pnl_distribution.png')

## 6. GARCH Forecast vs ATM Implied Vol

- **Risk Premium** (gold): Implied vol > GARCH forecast — market pays extra for vol insurance
- **Vol Surprise** (blue): GARCH forecast > Implied vol — realized vol exceeds what market priced


In [ ]:
from integration.garch_integration import run
garch_df = run()

In [ ]:
Image('../outputs/garch_vs_implied_vol.png')

## Summary

| Model | Strengths | Weaknesses |
|-------|-----------|------------|
| BSM | Closed-form, fast, well-understood | Flat vol — no smile/skew |
| Heston | Stochastic vol, captures smile/skew, semi-analytical | 5 params to calibrate, slower |

Key takeaways:
- Markets exhibit persistent **vol skew**: OTM puts trade at higher IV than OTM calls
- Delta-hedged positions show **gamma-theta balance** disrupted by real-world vol fluctuations
- GARCH forecasts typically sit below implied vol — the gap is the **variance risk premium**
